# 05 – Task 5: greedy speculative decoding
Target: fine-tuned `Qwen2.5-1.5B-Instruct` (the screening model). Draft: `Qwen2.5-0.5B-Instruct` (same tokenizer).
Step 1 fine-tunes the 0.5B draft with the same recipe as the target (~10 min), so we can compare a base draft
with a task-adapted draft. Step 2 runs the benchmark (~10 min). Code: `src/speculative.py`, `src/bench_spec.py`.

In [ ]:
# ==== EDIT THESE TWO LINES ====
GITHUB_USER = "FatimaRana50"
GIT_NAME    = "Fatima Rana"
# ==============================
REPO = "trialmatch"

!pip -q install peft datasets
!pip -q uninstall -y torchao       # Colab ships torchao 0.10; peft refuses to load with it (we do not use it)
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
token = userdata.get('GH_TOKEN')          # Colab sidebar -> key icon -> add secret GH_TOKEN
if not os.path.exists(f'/content/{REPO}'):
    !git clone -q https://{token}@github.com/{GITHUB_USER}/{REPO}.git /content/{REPO}
%cd /content/{REPO}
!git pull -q
!git config user.name "{GIT_NAME}"
!git config user.email "{GITHUB_USER}@users.noreply.github.com"
import sys; sys.path.insert(0, '/content/trialmatch/src')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
TARGET_ADAPTER = "/content/drive/MyDrive/trialmatch/lora_r16/final_adapter"
DRAFT_DIR = "/content/drive/MyDrive/trialmatch/draft_0.5b"
os.makedirs(DRAFT_DIR, exist_ok=True)

## 1. Fine-tune the 0.5B draft (train split only, val loss for checkpoint choice)

In [ ]:
!python src/train.py --base Qwen/Qwen2.5-0.5B-Instruct --out {DRAFT_DIR} --tag draft05

## 2. Speculative vs greedy: equality, acceptance, latency, throughput, memory

In [ ]:
!python src/bench_spec.py --target-adapter {TARGET_ADAPTER} --draft-adapter {DRAFT_DIR}/final_adapter

In [ ]:
import json, pandas as pd
from IPython.display import Image, display
r = json.load(open("results/spec_bench.json"))
print(f"speculative == greedy: {r['equal']}/{r['total']} prompt runs")
print(json.dumps(r["env"], indent=1))
cols = ["setting","method","draft","k","tokens","total_s","tok_s","speedup","acceptance_rate",
        "mean_accepted_per_round","draft_s","verify_s","target_forwards","greedy_target_forwards","equal_outputs","peak_mem_mb"]
print(pd.DataFrame(r["rows"]).reindex(columns=cols).round(3).to_string())
bad = [c for c in r["checks"] if not c["equal"]]
print("mismatches:", bad[:5])
display(Image("results/spec_bench.png"))

In [ ]:
!pip freeze > logs/colab_env_05.txt
!git add results/ logs/ && git commit -m "Speculative decoding benchmark (T4)" && git pull -q --rebase && git push